# 07 · Validation et promotion challenger → champion
Contrôle automatique sur le test (jamais vu pendant le tuning) : AUC ≥ seuil, et pas de régression
face au champion actuel. Si OK : alias `champion` déplacé et baseline de monitoring rafraîchie.

In [0]:
%pip install -q databricks-feature-engineering
%restart_python

In [0]:
import os, sys
sys.path.append(os.path.abspath("../src"))
import mlflow
from mlflow.tracking import MlflowClient
from pyspark.sql import functions as F
from databricks.feature_engineering import FeatureEngineeringClient
from churn.dbx import get_config
from churn.config import LABEL, TEST_SNAPSHOTS
from churn import modeling as M

cfg = get_config(dbutils)
fe = FeatureEngineeringClient()
mlflow.set_registry_uri("databricks-uc")
client = MlflowClient()
name = cfg.churn_model_name

# Le jeu de test : uniquement des clés + le label ; le Feature Store ira chercher les features
test_labels = (spark.table(cfg.t("attrition_labels"))
               .where(F.col("snapshot_date").between(*TEST_SNAPSHOTS)))


def score(alias):
    """Score le jeu de test avec la version du modèle désignée par l'alias."""
    preds = fe.score_batch(model_uri=f"models:/{name}@{alias}", df=test_labels, result_type="double")
    pdf = preds.toPandas()
    return pdf, M.evaluate(pdf[LABEL], pdf["prediction"])

In [0]:
challenger = client.get_model_version_by_alias(name, "challenger")
try:
    champion = client.get_model_version_by_alias(name, "champion")
except Exception:
    champion = None                         # 1re fois : pas encore de champion

if champion is not None and champion.version == challenger.version:
    dbutils.notebook.exit(f"La version {challenger.version} est déjà champion : rien à faire")

chal_pdf, chal_m = score("challenger")
champ_m = score("champion")[1] if champion is not None else None

checks = {
    "auc_au_dessus_du_seuil": chal_m["roc_auc"] >= cfg.min_auc,
    "pas_de_regression": champ_m is None or chal_m["roc_auc"] >= champ_m["roc_auc"] - cfg.max_auc_drop,
}
passed = all(checks.values())

print("Challenger :", {k: round(v, 3) for k, v in chal_m.items()})
print("Champion   :", None if champ_m is None else {k: round(v, 3) for k, v in champ_m.items()})
print("Contrôles  :", checks, "→", "VALIDÉ" if passed else "REJETÉ")

# Traçabilité : le résultat de la validation est écrit sur la version du modèle
for k, v in {**{f"check_{k}": str(v) for k, v in checks.items()},
             "validation_status": "PASSED" if passed else "REJECTED",
             "validation_test_auc": f"{chal_m['roc_auc']:.4f}"}.items():
    client.set_model_version_tag(name, challenger.version, k, v)

In [0]:
if passed:
    client.set_registered_model_alias(name, "champion", challenger.version)
    baseline = (chal_pdf.rename(columns={"prediction": "p_churn"})
                .assign(model_version=int(challenger.version)))
    (spark.createDataFrame(baseline).write.mode("overwrite")
     .option("overwriteSchema", "true").saveAsTable(cfg.t("monitoring_baseline")))
    print(f"✅ Version {challenger.version} promue champion ; baseline de monitoring écrite "
          f"({len(baseline)} lignes)")
else:
    print(f"⛔ Version {challenger.version} rejetée : le champion reste en place")